# 04 Investigate alerts

Read each explanation, then run the next cell with **Shift+Enter**. Add a code cell starting with `%%sql` for your own query. Save edits with **Ctrl+S / Cmd+S**. Tabular SQL results support sorting, search and pagination, with all returned rows available. Single-column nested results render as an expandable JSON tree. Use `%%sql --json` to inspect an entire result as JSON.

See [the participant guide](../WORKSHOP.md) for the full workshop.

## Connect to DuckDB

Run this setup once per kernel. It prepares an isolated DuckDB database with raw logs and reference relations for checking, sequence detection and investigation. Notebook 02 builds a separate participant-owned binding relation; reference ingestion does not complete that exercise for you. It also loads your saved participant rule, if present. SQL cells below run against this session; no cloud credentials or model key are required.

In [ ]:
from pathlib import Path
import sys
import os

root = Path.cwd().resolve()
while not (root / "workshop" / "jupyter.py").exists():
    if root == root.parent:
        raise RuntimeError("Open this notebook from the workshop checkout.")
    root = root.parent
sys.path.insert(0, str(root))
os.chdir(root)
from workshop.jupyter import NotebookSession
if "session" in globals():
    session.close()
session = NotebookSession()
get_ipython().run_line_magic("load_ext", "workshop.jupyter")
print("Ready. Run the SQL cells below with Shift+Enter.")

## Know the datasets before querying

The workshop separates **observed activity** and **context about that activity**. A schema gives them consistent field names; it does not
invent missing logs, resolve identities automatically, or prove malicious intent.

| Dataset | What it provides | How you will use it |
|---|---|---|
| `enterprise.raw_eks`, `enterprise.raw_gke`, `enterprise.raw_aks` | Modeled provider-specific audit records, with different JSON layouts and identity formats. | Inspect source evidence and understand what each mapping must preserve. |
| `enterprise.k8s_ocsf` | All Kubernetes activity in an OCSF-derived relational table: event ID, event time, provider/account/cluster, principal, operation, resource, success status and request data. One row represents an audit event, not an incident. | Write one behavioral sequence across clouds, retaining asset scope and event order. |
| `enterprise.k8s_subjects` | Child relation with one row per request subject, keyed by provider/account/cluster/event ID. | Follow every granted subject without duplicating rows in the event table. |
| `enterprise.identities` | Curated mappings from a native `principal_name` within its `cloud_provider` and `account_uid` to a shared `person_id`. | Connect differently formatted principals belonging to the same person. A LEFT JOIN retains unmapped activity; a missing person ID limits attribution, not detection. These links are supplied enrichment, not inferred by normalization. |
| `enterprise.approvals` | Maintenance tickets scoped to the grant actor, granted subject, role, provider/account/cluster and validity interval. | Suppress an otherwise matching sequence only when the exact change is approved at the grant time. A matching identity alone is not authorization. |

**SQL relations and workflow:**

| Relation or step | What it provides | How you will use it |
|---|---|---|
| `main.workshop_alerts` | Your editable detection query, defined as a SQL view. | Modify the behavioral rule and inspect its alerts. |
| `main.alert_evidence` | Supporting event IDs as individual rows. | Retrieve the source evidence for each alert. |
| `enterprise.k8s_ocsf` | All normalized Kubernetes event types, with one row per source event and asset. Role references are projected into columns when present. | Query normalized fields without repeating provider-specific JSON extraction. |
| `enterprise.k8s_subjects` | Repeated request subjects stored separately, preserving the event table's row count. | Join subjects to their source event to follow each granted subject. |
| Ingestion preparation | Relational fields prepared through engine-specific JSON extraction and file loading. | Keep preparation separate from the normalized detection SQL. |
| Notebook 01 | Three incomplete cloud-specific binding detections. | Write the predicates and check the matching source events. |
| Notebook 02 | A ten-field contract and participant-authored SQL mappings. | Build and validate your own binding table, then write one unified detection. |
| Notebook 03 | An advanced sequence over the fuller prepared schema. | Correlate grants with subsequent secret-list activity. |

**Workshop dataset:** `enterprise.*` uses the larger synthetic enterprise day:
routine controller and service-account traffic, deployments, human operations,
failed requests and suspicious activity in modeled EKS/GKE/AKS delivery formats.
Use provider, cluster and time filters to keep investigations manageable.
The cloud envelopes are modeled, not verified against real enterprise exports.


## SQL quick reference

Run each query, then use its `TODO` comment and **Try this** task to explore.
Single quotes delimit text; double quotes identify SQL columns, not strings.

| Task | Example |
|---|---|
| Match text exactly | `resource_type = 'clusterrolebindings'` |
| Extract text from raw JSON | `json_extract_string(event, '$.requestURI')` |
| Match part of a string | `LIKE '%clusterrolebinding%'` |
| Find a missing value | `person_id IS NULL` |
| Limit returned rows | `LIMIT 10` |
| Filter event time | `time >= TIMESTAMPTZ '2026-10-14 14:00:00+00'` |

Raw-source queries use JSON paths. Normalized queries use ordinary columns such as
`resource_type`, `principal` and `time`. Each SQL cell runs one statement.
Open **Need a hint?** only when you get stuck. An empty result can be valid after changing filters.


## Select an alert from your rule

List alerts produced by the current participant rule.

**What to look for:** Each row links an alert and its asset/subject keys to a supporting event_id and evidence_kind. Multiple evidence rows can belong to one alert; event bodies are not selected here.

**Try this:** Choose an alert and retain its alert_key, provider, account_uid, cluster_name and subject. Add filters on these keys inside the next query’s chosen CTE to investigate it instead of the default earliest alert.

In [2]:
%%sql
-- TODO: Choose an alert, keeping its asset and subject keys for the timeline below.
-- List alerts produced by the current participant rule.
-- Each row links an alert and its asset/subject keys to a supporting event_id and evidence_kind. Multiple evidence rows can belong to one alert; event bodies are not selected here.
SELECT a.alert_key, a.provider, a.account_uid, a.cluster_name,
       a.subject, a.grant_time, e.event_id, e.evidence_kind
FROM main.workshop_alerts a
JOIN main.alert_evidence e
  ON e.alert_key = a.alert_key AND e.provider = a.provider
 AND e.account_uid = a.account_uid AND e.cluster_name = a.cluster_name AND e.subject = a.subject
ORDER BY a.grant_time, e.time;

## Read the surrounding activity

Inspect the first alert’s actor and subject on the exact asset around the sequence.

**What to look for:** Inspect up to 100 events for the grant actor or subject on the same provider/account/cluster, ordered by time and event_id, from five minutes before the grant through five minutes after last_read. The cap can omit later events.

**Try this:** Filter the chosen CTE by your alert’s key, asset and subject, or adjust the time window and row limit. If no alerts exist, the timeline is empty. Missing events do not prove an action did not happen.

<details>
<summary>Need a hint?</summary>

Add filters inside chosen, before ORDER BY. Use the selected alert key together with provider/account/cluster and subject to retain scope.

</details>

In [10]:
%%sql
-- TODO: Select your chosen alert inside the CTE; adjust the window or row limit if needed.
-- Inspect the first alert’s actor and subject on the exact asset around the sequence.
-- Inspect up to 100 events for the grant actor or subject on the same provider/account/cluster, ordered by time and event_id, from five minutes before the grant through five minutes after last_read. The cap can omit later events.
WITH chosen AS (
    SELECT * FROM main.workshop_alerts
    ORDER BY grant_time LIMIT 1
)
-- Retrieve activity for the grant actor and subject on this exact asset.
-- Include five minutes around the sequence and cap output at 100 events.
SELECT e.* FROM enterprise.k8s_ocsf e JOIN chosen c
 ON e.provider = c.provider AND e.account_uid = c.account_uid AND e.cluster_name = c.cluster_name
AND e.time BETWEEN c.grant_time - INTERVAL '5' MINUTE AND c.last_read + INTERVAL '5' MINUTE
WHERE e.principal IN (c.grant_actor, c.subject)
ORDER BY e.time, e.event_id FETCH FIRST 100 ROWS ONLY;

## Explain why this alert fired — 3 minutes

Before running the scripted checker or AI investigator, use the **source-event timeline above** to decide whether this alert matches the rule you wrote. Your goal is to explain the observed behavior, not to declare an incident.

### Check the two sides of the sequence

| Check | Evidence to find |
|---|---|
| Access was granted | A successful `create` of a `clusterrolebindings` resource, with `role_kind = 'ClusterRole'`, `role = 'cluster-admin'` and the recipient listed as a User in the request subjects |
| The recipient used that access | A successful `list` of `secrets` by that recipient, with `namespace IS NULL` for a cluster-wide request |
| The events belong together | The same provider, account and cluster; the secret list occurs at or after the grant, within 15 minutes |

The actor creating the binding may differ from the recipient. Follow the **recipient** when checking the secret-list event. Use `status_id = 1` to verify success. Copy the event IDs so someone else can check your finding.

### Write your finding

Add a **Markdown cell** below this section (press Escape, then B to insert a cell, then M to make it Markdown). Fill in these five lines using your selected alert:

```text
Grant event ID: ...
Secret-list event ID: ...
Why the rule matched: [actor] granted [recipient] cluster-admin on [provider/account/cluster] at [time]; [recipient] listed secrets at [time], [N] minutes later.
Possible legitimate explanation: ...
Before escalating, I would check: ...
```

For the last two lines, consider whether this was expected administrative work and which ticket, owner confirmation or additional logs would help establish that. The exercise data does not supply validated approvals; an empty approvals table does not prove the work was unauthorized.

**You are done when:** your finding cites both events and explains the recipient, asset scope, success and time window. If you cannot find a required event, say which evidence is missing instead of filling the gap with an assumption. A successful secret-list request alone does not establish that secrets were exfiltrated.

Next, run the scripted checker and compare its claims with your own finding. Check any disagreement against the events; do not treat its disposition as the answer key.

## Run the scripted evidence checker

Choose an alert from your current rule. This checker follows a fixed Python sequence to retrieve the grant, subsequent subject activity, approvals and inventory. It makes no LLM calls. Inspect its returned rows and severity reasons; its hard-coded assessment is a suggestion to review, not proof of compromise. The import `from agent.triage import run` points to the shared Python module: calling `run` without `model` selects this scripted mode.

In [4]:
from workshop.exercises import artifacts, save_records
from agent.triage import run
import pandas as pd
session.refresh_rule()
alerts = session.execute("SELECT DISTINCT alert_key FROM main.workshop_alerts ORDER BY 1")
# TODO: keep the first alert, or uncomment the next line and paste another alert key.
selected_alert_key = alerts[0]["alert_key"] if alerts else None
# selected_alert_key = "COPY_ALERT_KEY"
if selected_alert_key:
    own_record = run(session.engine, selected_alert_key, split="enterprise")
    save_records("my-notebook-alert.jsonl", [own_record])
    display(own_record["assessment"])
    display(pd.DataFrame(own_record["trace"]))
else:
    print("Your rule produced no alerts. Review the source activity and rule conditions.")

{'disposition': 'malicious',
 'severity': 'critical',
 'severity_reasons': ['Threat-intelligence-listed source IP (+1)',
  'Restricted production cluster (+1)'],
 'summary': 'Scripted replay of evidence checks; this is not an LLM-generated finding.',
 'attribution': 'resolved',
 'claims': [{'kind': 'rbac_grant',
   'event_id': '0ba32758-91d5-4f85-f45f-7b1c54ebc279',
   'explanation': 'Successful cluster-admin grant observed.'},
  {'kind': 'secret_list',
   'event_id': '80c484c2-a579-c3c2-154a-3c725f8c2558',
   'explanation': 'Successful cluster-wide secret listing after the grant.'}],
 'benign_alternatives': ['Consider legitimate administrative work before operational escalation'],
 'uncertainties': [],
 'follow_up_queries': []}

tool                                              input  \
0               get_alert  {'event_id': '0ba32758-91d5-4f85-f45f-7b1c54eb...   
1  get_cluster_enrichment  {'provider': 'AWS', 'account_uid': '1111222233...   
2     get_user_enrichment  {'provider': 'AWS', 'account_uid': '1111222233...   
3        resolve_identity  {'provider': 'AWS', 'account_uid': '1111222233...   
4      get_actor_timeline  {'provider': 'AWS', 'account_uid': '1111222233...   
5     get_change_approval  {'provider': 'AWS', 'account_uid': '1111222233...   
6       get_ip_enrichment                            {'ip': '198.51.100.23'}   
7       submit_assessment  {'disposition': 'malicious', 'severity': 'crit...   

                                              result  is_error  latency_ms  
0  {'rows': [{'event_id': '0ba32758-91d5-4f85-f45...     False          41  
1  {'rows': [{'cloud_provider': 'AWS', 'account_u...     False          33  
2  {'rows': [{'cloud_provider': 'AWS', 'account_u...     False          33  
3  {'rows': [{'person_id': 'P-1001', 'cloud_provi...     False          32  
4  {'rows': [{'event_id': 'fe0c4328-1b4e-1ef5-f96...     False          38  
5                   {'rows': [], 'truncated': False}     False          32  
6  {'rows': [{'ip_address': '198.51.100.23', 'ip_...     False          33  
7  {'disposition': 'malicious', 'severity': 'crit...     False           0

## Optionally run an AI investigator with a local model

This optional mode uses an LLM to choose evidence tools and write an assessment. Passing `model=LOCAL_MODEL` to `run` selects this mode. Inspect every claim and returned event; there is no ground-truth score in this workshop.

### Install and start Ollama

Run these commands in a **terminal**, on the same machine as the Jupyter kernel.

1. Install Ollama from [ollama.com/download](https://ollama.com/download). On macOS or Windows, open the installed Ollama application. On Linux, install it with:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```
2. If Ollama is not already running as an application or service, start it and leave this terminal open:
   ```bash
   ollama serve
   ```
   If it reports that port 11434 is already in use, Ollama may already be running. Check with `ollama list` in another terminal.
3. In another terminal, download the model:
   ```bash
   ollama pull qwen3:8b
   ```

### Run the AI investigator

Run the **scripted evidence checker** cell above first; it imports `run` and selects `selected_alert_key`. Then change the settings in the next cell to:

```python
RUN_LOCAL_MODEL = True
LOCAL_MODEL = "qwen3:8b"
```

Run that cell with **Shift+Enter**. It connects to Ollama at `http://localhost:11434`, displays the assessment and tool trace, and saves the record to `artifacts/my-local-model-alert.jsonl`. No API key is needed. The initial model download requires internet access; model execution may take several minutes, depending on your hardware.

Leave `RUN_LOCAL_MODEL = False` to skip the AI investigator and use only the scripted checker.

In [5]:
# TODO: enable local inference only after installing Ollama and downloading this model.
RUN_LOCAL_MODEL = True
LOCAL_MODEL = "qwen3:8b"
if RUN_LOCAL_MODEL and selected_alert_key:
    model_record = run(session.engine, selected_alert_key, split="enterprise", model=LOCAL_MODEL)
    save_records("my-local-model-alert.jsonl", [model_record])
    display(model_record["assessment"])
    display(pd.DataFrame(model_record["trace"]))
else:
    print("Local inference skipped. Enable RUN_LOCAL_MODEL after installing Ollama.")

RuntimeError: Local Ollama request failed. Start `ollama serve` and pull the selected model. No paid provider was contacted.

## Compare raw and normalized evidence with the same AI investigator

**Question:** Does a shared schema reduce the translation work an LLM must do to investigate the same behavior?

This is an optional extension after the one-hour core workshop. You run it yourself; no facilitator or saved model answers are required. Complete the Ollama setup above first. Allow several minutes per run, and potentially much longer on a CPU.

| Setting | Raw run | Normalized run |
|---|---|---|
| Task | Investigate the selected grant and cite supporting events | Same task and grant |
| Evidence | `raw_eks`, `raw_gke`, `raw_aks`, with provider-specific JSON | One `events` table with consistent event fields and request data |
| Tools | `list_tables`, `query_sql`, `submit_assessment` | Identical tools |
| Model, prompt, turn limit | Your local model, same investigation prompt, 12 turns | Identical settings, fresh conversation |
| Supporting context | Identity, inventory and approval tables | Same context |

Guided lookup tools would hide schema differences, so **both runs use SQL-only tools**. The normalized snapshot comes from the fuller prepared event schema, not the small binding-only table you built in notebook 02. Normalized request subjects remain in request data in this snapshot; the model may still need JSON extraction.

### Run your first pair

1. Run setup and the scripted checker above to define `session` and `selected_alert_key`.
2. In the next cell, set `RUN_SCHEMA_COMPARISON = True` and use the model you downloaded. You can skip the earlier single AI run; this comparison makes its own model calls.
3. Run the cell. It starts **two fresh investigations of the same grant**, raw then normalized. Progress appears before each run. Each completed result or error is saved to `artifacts/raw-vs-normalized.jsonl`; rerunning replaces that file.
4. Read the comparison table and the query/assessment output below it. `submitted` means the model returned an assessment, not that its answer is correct. `incomplete` means it exhausted the turn limit; `error` usually means the local service or model request failed.

Reference-event coverage is calculated **after** each run using your rule's supporting events; these reference IDs are not shown to the model. Coverage is a navigation aid, not a correctness score. Check cited event contents, asset scope, recipient, success and the 15-minute window yourself. A model may retrieve an event without reasoning about it correctly.

### Compare what the model had to do

- Did both assessments cite the grant and a valid secret-list event? Did either cite events it never retrieved?
- In the raw queries, find provider table choices, JSON paths and native status checks. What replaces them in the normalized queries? Tool errors count SQL/tool failures, not silent reasoning mistakes.
- Which run needed fewer SQL queries or corrections? Compare token usage only after checking the evidence.
- Is the conclusion stronger than the logs support? Secret-list metadata alone does not prove exfiltration or malicious intent.

To explore all three clouds, set `COMPARE_ALL_CLOUDS = True` and rerun. This chooses one alert per provider present in your rule, for **up to six model runs**. Each pair uses the same alert; it does not merge cloud activity or prove cross-cloud attribution. A single-provider pair demonstrates field translation; the three-provider extension shows how those translation differences repeat.

Results may favor either representation. Timing excludes snapshot creation but includes model/tool work and model loading. Raw runs first, so model warm-up can affect timing; repeat in the opposite order with `NORMALIZED_FIRST = True` before drawing timing conclusions. One small experiment does not establish a general performance advantage.

**Finish:** Write two sentences describing a query difference you observed and whether both answers were supported by evidence. If both runs fail, inspect the errors, verify `ollama list`, and try again with the same settings for both runs. Do not interpret failure as evidence that one schema is better.

In [ ]:
from workshop.agent_comparison import compare_evidence, comparison_rows
import pandas as pd
from IPython.display import display

# TODO: install/start Ollama and pull this model using the instructions above.
RUN_SCHEMA_COMPARISON = False
COMPARISON_MODEL = "qwen3:8b"
COMPARE_ALL_CLOUDS = False
NORMALIZED_FIRST = False

if RUN_SCHEMA_COMPARISON:
    comparison_records = compare_evidence(
        session, selected_alert_key, COMPARISON_MODEL,
        all_clouds=COMPARE_ALL_CLOUDS, max_turns=12,
        normalized_first=NORMALIZED_FIRST,
    )
    display(pd.DataFrame(comparison_rows(comparison_records)))
    for record in comparison_records:
        print("\n", record["comparison_target"], record["representation"])
        if record.get("error"):
            print(record["error"])
        for step in record["trace"]:
            if step["tool"] == "query_sql":
                print("SQL:", step["input"].get("sql"))
                if step["is_error"]:
                    print("ERROR:", step["result"])
        display(pd.DataFrame(record["trace"]))
        display(record.get("assessment") or "No assessment submitted; inspect the saved trace.")
else:
    print("Comparison skipped. Set RUN_SCHEMA_COMPARISON=True after the Ollama setup.")

You have completed the core notebook workflow. Continue with the optional extra lab [05 Run the detection in batches](05_Run_the_detection_in_batches.ipynb), or review your artifacts and record the rule specification, source evidence and remaining uncertainty. See [the agent guide](../docs/agent-lab.md) for additional background.